In [1]:
import pandas as pd 
import numpy as np 
import os 
from scipy.fft import fft, fftfreq
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.io import loadmat
from scipy.signal import windows, periodogram,welch, spectrogram
import numpy as np
import mne
os.makedirs(os.path.join(os.getcwd(),'images'),exist_ok=True)
os.makedirs(os.path.join(os.getcwd(),'datasets'),exist_ok=True)
IMAGE_DIR = os.path.join(os.getcwd(),'images')
DATASETS = os.path.join(os.getcwd(),'datasets')

# Load the eeg_data.mat dataset and generate the graph as illustrated in the notes.

### Uncomment the appropriate section of code to load the corresponding dataset.
### Each file type (.mat, .txt, .csv) is processed using its respective method.


In [2]:
# EEG data from class

# data = loadmat(os.path.join(os.getcwd(),'datasets','eeg_data.mat')) 
# eeg_signal = data['eeg'].squeeze() # squeeze() is used to remove single-dimensional entries from the shape of an array.
# sampling_frequency = 50  # Sampling frequency
# signal_name = 'eeg_data'


# data = loadmat(os.path.join(os.getcwd(),'datasets','S05E.mat')) 
# eeg_signal = data['data'].squeeze() # squeeze() is used to remove single-dimensional entries from the shape of an array.
# sampling_frequency = 512  # Sampling frequency
# signal_name = 'S05E'

# data = loadmat(os.path.join(os.getcwd(),'datasets','Hr_pre.mat')) 
# eeg_signal = data['hr_pre'] # squeeze() is used to remove single-dimensional entries from the shape of an array.
# sampling_frequency = 1  # Sampling frequency
# signal_name = 'ecg'



#EMOTION
# electrodes = pd.read_csv(os.path.join(DATASETS, "10_18.0.electrodes.txt"),sep="\t",  )['labels'].tolist()
# df = pd.read_csv(os.path.join(DATASETS, "10_10.0.txt"),sep="\t", names=electrodes )
# eeg_signal = df['F8'].values
# sampling_frequency = 256  # Sampling frequency
# signal_name = 'emotion'



# # # MUSE
# df = pd.read_csv(os.path.join(DATASETS,'csh_30_s1.csv'))
# df.sort_values('timestamps',inplace=True)
# eeg_signal = df['TP10'].values
# # signal = df.loc[df['timestamps']>=1644826281]['AF7'].values
# sampling_frequency = 220  # Sampling frequency
# signal_name = 'muse'

# EEG Motor Movement/Imagery
#  https://physionet.org/content/eegmmidb/1.0.0/S001/#files-panel
# data = mne.io.read_raw_edf(os.path.join(DATASETS,'S001R03.edf'))
# eeg_signal = data.copy().pick(['C4..']).get_data()[0]
# sampling_frequency = data.info['sfreq']
# signal_name = 'movement_imagery'

# import mne
# data = mne.io.read_raw_eeglab(os.path.join(DATASETS,'sub-048_task-eyesclosed_eeg.set'), preload=True)
# sampling_frequency = data.info['sfreq']
# eeg_signal = data.copy().pick_channels(['Cz']).get_data()[0]
# signal_name = 'dementia_cn'



# data = mne.io.read_raw_edf(os.path.join(DATASETS,'Subject01_1.edf'))
# eeg_signal = data.get_data(picks=['EEG O1'])[0]
# sampling_frequency = data.info['sfreq']
# signal_name = 'arithmetic_task_subject_1_s1'



data = mne.io.read_raw_edf(os.path.join(DATASETS,'sub-001_ses-01_task-szMonitoring_run-01_emg.edf'))
eeg_signal = data.get_data(picks=['EMG SD'])[0][:32659]
sampling_frequency = data.info['sfreq']
signal_name = 'emg'

Extracting EDF parameters from /home/volpym/Dropbox/Notes/MSC/Biosignal Processing - Neuroinformatics/projects/datasets/sub-001_ses-01_task-szMonitoring_run-01_emg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...


In [4]:
time = np.arange(1, len(eeg_signal) ) * (1 / sampling_frequency)  # Time vector

In [5]:
time

array([3.90625000e-03, 7.81250000e-03, 1.17187500e-02, ...,
       1.27562500e+02, 1.27566406e+02, 1.27570312e+02], shape=(32658,))

In [6]:
fig = go.Figure()

fig.add_trace(go.Scatter(

    x=time,
    y=eeg_signal,
    mode='lines',
    name='EEG trace'
))

fig.update_layout(
    title='EEG trace',
    xaxis_title='Time (s)',
    yaxis_title='Ampitude (V)',
    #yaxis=dict(range=[np.min(eeg_signal), np.max(eeg_signal)]),  # Equivalent to xlim([0 16])
    template='plotly_white',
    width=2000, height=500
)

fig.write_image(os.path.join(IMAGE_DIR,f"{signal_name}_trace.png"), width=1800, height=500, scale=2)

fig.show()


# Apply FFT (Fast Fourier Transform) to convert the signal to the frequency domain

In [23]:
# Remove DC component (mean)
eeg_signal = eeg_signal - np.mean(eeg_signal)
N = len(eeg_signal)
X = fft(eeg_signal)
two_sided = np.abs(X) / N
one_sided = two_sided[:N//2 + 1]
one_sided[1:-1] *= 2             # double non‑DC, non‑Nyquist bins

freqs = sampling_frequency * np.arange(0, N//2 + 1) / N
f = sampling_frequency * np.arange(0, len(eeg_signal)//2 + 1) / len(eeg_signal)


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=f,
    y=one_sided,
    mode='lines',
    name='EEG trace'
))

fig.update_layout(
    title='Single-Sided Amplitude Spectrum',
    xaxis_title='f (Hz)',
    yaxis_title='Magnitude (μV)',
    #xaxis=dict(range=[0, 50]),  # Equivalent to xlim([0 16])
    #yaxis=dict(range=[0, 10]),  # Equivalent to xlim([0 16])
    template='plotly_white'
)
fig.write_image(os.path.join(IMAGE_DIR,f"{signal_name}_amplitude_spectrum.png"), width=800, height=500, scale=2)
fig.show()


In [ ]:
faxis, pxx = periodogram(
    eeg_signal,
    fs=sampling_frequency,
    window=windows.hamming(len(eeg_signal)),
    nfft=len(eeg_signal),
    scaling='density',
    return_onesided=True
)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=faxis,
    y=pxx,
    mode='lines',
    name='EEG trace'
))

fig.update_layout(
    title='periodogram',
    xaxis_title='f (Hz)',
    yaxis_title='|P(db)|',
    xaxis=dict(range=[0, 50]),  # Equivalent to xlim([0 16])
    template='plotly_white',
    
)
fig.write_image(os.path.join(IMAGE_DIR,f"{signal_name}_periodogram.png"), width=800, height=500, scale=2)
fig.show()

ValueError: the size of the window must be the same size of the input on the specified axis

In [362]:

WINDOW = 4 *int(sampling_frequency)         
NOVERLAP = WINDOW//2       
NFFT = WINDOW          
window = windows.hamming(WINDOW)  

faxis, pxx = welch(
    eeg_signal,
    fs=sampling_frequency,
    window=window,
    nperseg=WINDOW,
    noverlap=NOVERLAP,
    nfft=NFFT,
    scaling='density',
    return_onesided=True
)
#pxx_db = 10 * np.log10(pxx)
fig = go.Figure()
fig.add_trace(go.Scatter(x=faxis, y=pxx, mode='lines'))
fig.update_layout(
    xaxis_title='f (Hz)',
    template='plotly_white',
    
)
fig.update_xaxes(range=[0, 40])
fig.write_image(os.path.join(IMAGE_DIR,f"{signal_name}_pwelch_{NOVERLAP}_samples.png"), width=800, height=500, scale=2)
fig.show()

In [354]:
WINDOW

2000

In [343]:
WINDOW = 1000
NOVERLAP = WINDOW//2
NFFT = WINDOW

# Create a Hamming window
window = windows.hamming(WINDOW)

# Compute Welch PSD
faxis, pxx = welch(
    eeg_signal,
    fs=sampling_frequency,
    window=window,
    nperseg=WINDOW,
    noverlap=NOVERLAP,
    nfft=NFFT,
    return_onesided=True,  # 'onesided' in MATLAB
    scaling='density'      # PSD in power/Hz
)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=faxis,
    y=pxx,
    mode='lines',
    name='EEG trace'
))

fig.update_layout(
    title=f'pwelch {NOVERLAP} samples overlap',
    xaxis_title='f (Hz)',
    yaxis_title='|P(db)|',
    xaxis=dict(range=[0, 20]),  # Equivalent to xlim([0 16])
    template='plotly_white'
)
fig.write_image(os.path.join(IMAGE_DIR,f"{signal_name}_pwelch_{NOVERLAP}_samples.png"), width=800, height=500, scale=2)
fig.show()

In [344]:
WINDOW = 256
NOVERLAP = 255
NFFT = 256
# Create a Hamming window
window = windows.hamming(WINDOW)

# Compute Welch PSD
faxis, pxx = welch(
    eeg_signal,
    fs=sampling_frequency,
    window=window,
    nperseg=WINDOW,
    noverlap=NOVERLAP,
    nfft=NFFT,
    return_onesided=True,  # 'onesided' in MATLAB
    scaling='density'      # PSD in power/Hz
)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=faxis,

# Continuous Wavele
    y=pxx,
    mode='lines',
    name='EEG trace'
))

fig.update_layout(
    title=f'pwelch {NOVERLAP} samples overlap',
    xaxis_title='f (Hz)',
    yaxis_title='|P(db)|',
    #xaxis=dict(range=[0, 1]),  # Equivalent to xlim([0 16])
    template='plotly_white',
)
fig.write_image(os.path.join(IMAGE_DIR,f"{signal_name}_pwelch_{NOVERLAP}_samples.png"), width=800, height=500, scale=2)
fig.show()


In [393]:
f, t, Sxx = spectrogram(eeg_signal, fs=sampling_frequency, nperseg=128, noverlap=50)
normalized_freq = f / np.max(f)
fig = go.Figure()

fig.add_trace(
    go.Heatmap(
        x=t,              # use corrected time base
        y=f,
        z=Sxx,
        colorscale='Viridis',
        colorbar=dict(title='Power (dB)', orientation='h'),
        name='Spectrogram'
    ),
)

fig.update_layout(
    height=800,
    width=1500,
    showlegend=True,
    xaxis_title='Time(s)',
    yaxis_title='Frequency(Hz)',
)
fig.write_image(os.path.join(IMAGE_DIR,f"{signal_name}_spectogram.png"), width=600, height=500, scale=2)
fig.show()
